# Stage 6 – Network measures

**Goal:** compute the node-level and network-level measures for each format with one shared set of rules
(see `library/lib_05_network_measures.ipynb` and `docs/methods_decisions.md`).

**Reads:** `data/processed/nodes_<format>.csv`, `edges_<format>.csv`
**Writes:** `outputs/tables/node_metrics_<format>.csv`, `network_summary.csv`, `communities_<format>.csv`,
`core_sizes_<format>.csv`

This stage takes about a minute, because of the 200 random graphs and the 100 Louvain runs per format.

In [1]:
%run ./library/lib_00_settings.ipynb
%run ./library/lib_04_network_building.ipynb
%run ./library/lib_05_network_measures.ipynb
stage_log = get_stage_logger("06_metrics")

In [2]:
summary_by_format = {}
for format_name in FORMATS:
    file_tag = FORMAT_FILE_TAG[format_name]
    partnership_graph = load_format_graph(format_name)
    node_measure_table = node_measures(partnership_graph)
    write_table(node_measure_table, TABLES_FOLDER / f"node_metrics_{file_tag}.csv")
    summary_by_format[format_name] = network_summary(partnership_graph, node_measure_table)

    # How many batters sit in each k-core shell.
    core_sizes = node_measure_table.groupby("core_number").size().rename("players").reset_index()
    core_sizes.insert(0, "format", format_name)
    write_table(core_sizes, TABLES_FOLDER / f"core_sizes_{file_tag}.csv")

    # Members of each Louvain community, strongest batter first.
    community_table = (node_measure_table.sort_values(["community", "strength"], ascending=[True, False])
                       .groupby("community")
                       .agg(players=("player_id", "size"), total_node_strength=("strength", "sum"),
                            members=("player_name", lambda names: "; ".join(names)))
                       .reset_index())
    community_table.insert(0, "format", format_name)
    write_table(community_table, TABLES_FOLDER / f"communities_{file_tag}.csv")
    stage_log.info("%s: %s", format_name, {key: (round(value, 4) if isinstance(value, float) else value)
                                           for key, value in summary_by_format[format_name].items()})

network_summary_table = pd.DataFrame(summary_by_format).rename_axis("measure").reset_index()
write_table(network_summary_table, TABLES_FOLDER / "network_summary.csv")

2026-11-16 20:50:18,346 INFO batnet.06_metrics: Test: {'nodes': 51, 'edges': 335, 'density': 0.2627, 'components': 1, 'lcc_nodes': 51, 'lcc_share': 1.0, 'connected': 1, 'degree_mean': 13.1373, 'degree_median': 12.0, 'degree_min': 1, 'degree_max': 38, 'degree_sd': 8.7316, 'degree_centralization': 0.5176, 'strength_mean': 1689.4118, 'strength_median': 623.0, 'strength_max': 15303, 'edge_weight_total': 43080, 'edge_weight_mean': 128.597, 'edge_weight_median': 42.0, 'edge_weight_max': 2697, 'edges_single_observation_share': 0.391, 'top10pct_edges_weight_share': 0.6273, 'avg_clustering': 0.6164, 'transitivity': 0.5174, 'avg_path_length_lcc': 1.8886, 'diameter_lcc': 4, 'degree_assortativity': -0.1641, 'max_core_k': 11, 'max_core_nodes': 20, 'louvain_communities': 5, 'louvain_modularity': 0.2241, 'louvain_sizes': '24|14|7|4|2', 'mean_effective_size': 6.2908, 'mean_efficiency': 0.4733, 'mean_constraint': 0.2472, 'median_innings_batted': 18, 'C_random': 0.2616, 'L_random': 1.7575, 'C_ratio': 2.

2026-11-16 20:50:19,289 INFO batnet.06_metrics: ODI: {'nodes': 50, 'edges': 262, 'density': 0.2139, 'components': 1, 'lcc_nodes': 50, 'lcc_share': 1.0, 'connected': 1, 'degree_mean': 10.48, 'degree_median': 9.0, 'degree_min': 1, 'degree_max': 24, 'degree_sd': 6.8071, 'degree_centralization': 0.2874, 'strength_mean': 917.16, 'strength_median': 427.5, 'strength_max': 4837, 'edge_weight_total': 22929, 'edge_weight_mean': 87.5153, 'edge_weight_median': 40.0, 'edge_weight_max': 953, 'edges_single_observation_share': 0.5076, 'top10pct_edges_weight_share': 0.487, 'avg_clustering': 0.5266, 'transitivity': 0.4667, 'avg_path_length_lcc': 2.0433, 'diameter_lcc': 4, 'degree_assortativity': -0.0565, 'max_core_k': 9, 'max_core_nodes': 23, 'louvain_communities': 4, 'louvain_modularity': 0.2892, 'louvain_sizes': '19|15|8|8', 'mean_effective_size': 5.7458, 'mean_efficiency': 0.5601, 'mean_constraint': 0.2746, 'median_innings_batted': 11.0, 'C_random': 0.2142, 'L_random': 1.8663, 'C_ratio': 2.4588, 'L_r

2026-11-16 20:50:19,972 INFO batnet.06_metrics: T20I: {'nodes': 44, 'edges': 222, 'density': 0.2347, 'components': 1, 'lcc_nodes': 44, 'lcc_share': 1.0, 'connected': 1, 'degree_mean': 10.0909, 'degree_median': 10.0, 'degree_min': 1, 'degree_max': 27, 'degree_sd': 7.021, 'degree_centralization': 0.412, 'strength_mean': 862.4545, 'strength_median': 248.5, 'strength_max': 6281, 'edge_weight_total': 18974, 'edge_weight_mean': 85.4685, 'edge_weight_median': 30.5, 'edge_weight_max': 1397, 'edges_single_observation_share': 0.4054, 'top10pct_edges_weight_share': 0.5018, 'avg_clustering': 0.5494, 'transitivity': 0.5019, 'avg_path_length_lcc': 1.9588, 'diameter_lcc': 4, 'degree_assortativity': -0.1784, 'max_core_k': 8, 'max_core_nodes': 22, 'louvain_communities': 4, 'louvain_modularity': 0.1883, 'louvain_sizes': '23|10|10|1', 'mean_effective_size': 5.1376, 'mean_efficiency': 0.5365, 'mean_constraint': 0.3065, 'median_innings_batted': 9.5, 'C_random': 0.2362, 'L_random': 1.8338, 'C_ratio': 2.3256

### Network-level summary

In [3]:
network_summary_table

,measure,Test,ODI,T20I
0,nodes,51,50,44
1,edges,335,262,222
2,density,0.262745,0.213878,0.234672
3,components,1,1,1
4,lcc_nodes,51,50,44
5,lcc_share,1.0,1.0,1.0
6,connected,1,1,1
7,degree_mean,13.137255,10.48,10.090909
8,degree_median,12.0,9.0,10.0
9,degree_min,1,1,1
